# Combined Requirements Data Analysis
This notebook combines data from two Google Sheets tabs:
- Tab 0: "2025 +" (current/recent data)
- Tab 1: "DITM & Crab Trap MASTER Report (Through 2024)" (historical data)

It generates updated versions of all 4 requirements charts plus a new requirements vs absorption comparison chart.

In [1]:
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime
import os
from aquila_graphing_tools import initialize_supabase_connection, AQUILA_COLORS, AQUILA_FONT

In [3]:
# Connect to Google Sheets
scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
credentials = ServiceAccountCredentials.from_json_keyfile_name('aquilacommercialsheets-923494a59a4b.json', scope)
client = gspread.authorize(credentials)

spreadsheet_id = '1bzpRnUrpBH6l_zX7DtTypczZf5bpYVwqPUG3tzg2vec'
sheet = client.open_by_key(spreadsheet_id)

print(f"Connected to Google Sheets: {sheet.title}")
print(f"Available worksheets: {[ws.title for ws in sheet.worksheets()]}")

Connected to Google Sheets: DITM & Crab Trap - 2.0
Available worksheets: ['2025 +', 'Copy of 2025', 'DITM & Crab Trap MASTER Report (Through 2024)', 'Guidelines', 'Summary', 'test run', 'test run (1)']


In [4]:
# Read Tab 0: "2025 +" (current data)
tab0 = sheet.get_worksheet(0)
df_2025_plus = pd.DataFrame(tab0.get_all_records())

print(f"Tab 0 - '2025+':")
print(f"  Rows: {len(df_2025_plus)}")
print(f"  Columns: {list(df_2025_plus.columns)}")
print(f"\nFirst few rows:")
df_2025_plus.head(3)

Tab 0 - '2025+':
  Rows: 739
  Columns: ['DATE OF REQUIREMENT', 'DATE UPDATED', 'TENANT REP COMPANY', 'TENANT REP BROKER CONTACT', 'DEAL NAME', 'USE', 'MARKET', 'REQUIRED SF (LOW)', 'REQUIRED SF (HIGH)', 'TIMING', 'COMMENTS', 'INDUSTRY', 'ABSORBTION STATUS', 'ASSIGNED BROKER', 'BUILDINGS SENT', 'SOURCE', 'INTERNAL NOTES', 'Timing', 'Timing Trim', 'CRAB TRAP/DITM', 'STATUS']

First few rows:


,DATE OF REQUIREMENT,DATE UPDATED,TENANT REP COMPANY,TENANT REP BROKER CONTACT,DEAL NAME,USE,MARKET,REQUIRED SF (LOW),REQUIRED SF (HIGH),TIMING,...,INDUSTRY,ABSORBTION STATUS,ASSIGNED BROKER,BUILDINGS SENT,SOURCE,INTERNAL NOTES,Timing,Timing Trim,CRAB TRAP/DITM,STATUS
0,1/22/26,1/22/26,Fortune International Realty,Jorde Lluch,AI Tech Company,Office,C,27000,32000,,...,Technology,New to Market,"Bart, Chad, Seth",Zilker Point,Direct Email/Call,,,,DITM,Active
1,1/21/26,1/21/26,JLL,Mark Harris & Elizabeth Thompson,,Office,"S, E",12500,17500,Q1 2027,...,,,"Seth, Cody","Penn Field, Alto, Eastlake at Tillery",Direct Email/Call,,,,Crab Trap,Active
2,1/21/26,1/21/26,Colliers,Stuart Baker,Hair Salon,Retail,N,1500,2000,,...,Other,New to Market,"Seth, Cody",The Village,Direct Email/Call,,,,DITM,Active


In [20]:
# Read Tab 1: "DITM & Crab Trap MASTER Report (Through 2024)" (historical data)
tab1 = sheet.get_worksheet(2)

# Use get_all_values (faster, more robust for large sheets), then apply pd.DataFrame
rows = tab1.get_all_values()
df_through_2024 = pd.DataFrame(rows[1:], columns=rows[0])  # skip header row for data

# Select only rows where the "USE MARKET" column contains 'office' (case-insensitive)
if "USE" in df_through_2024.columns:
    df_through_2024 = df_through_2024[df_through_2024["USE"].str.lower().str.contains("office", na=False)]

print(f"Tab 1 - 'Through 2024':")
print(f"  Rows: {len(df_through_2024)}")
print(f"  Columns: {list(df_through_2024.columns[:15])}...")  # Show first 15 columns
print(f"\nFirst few rows:")
df_through_2024.head(3)

Tab 1 - 'Through 2024':
  Rows: 6971
  Columns: [' ', 'CRAB TRAP/DITM', 'DATE OF REQUIREMENT', 'DATE UPDATED', 'DEAL NAME', 'USE', 'MARKET', 'REQUIRED SF (LOW)', 'REQUIRED SF (HIGH)', 'TIMING', 'TENANT REP COMPANY', 'TENANT REP BROKER CONTACT', 'LEAD BROKER', 'COMMENTS', 'INTERNAL NOTES']...

First few rows:


,,CRAB TRAP/DITM,DATE OF REQUIREMENT,DATE UPDATED,DEAL NAME,USE,MARKET,REQUIRED SF (LOW),REQUIRED SF (HIGH),TIMING,TENANT REP COMPANY,TENANT REP BROKER CONTACT,LEAD BROKER,COMMENTS,INTERNAL NOTES,,
0,Active,DITM,11/26/24,,TileBar,"Office, Retail",C,"5,000","10,000",,Rise Commercial Partners,Tony Okelberry,,Hartland Plaza,,,
1,Done,DITM,11/25/24,,CodiumAI/ Qodo,Office,"E, S","20,000","25,000",March of 2025,Tower Commercial,Bill Gump and Susannah Davis,,Alto and MO. Want 2-3 year term or sublease. A...,Signed sublease at Shoal Creek Walk,,
2,Active,Crab Trap,11/25/24,,,Office,NW,"2,000","4,000",,Newmark,Joshua LaFico,,9500 Parmer,,,


In [21]:
# Analyze column overlap
cols_2025 = set(df_2025_plus.columns)
cols_2024 = set(df_through_2024.columns)

common = cols_2025.intersection(cols_2024)
only_2025 = cols_2025 - cols_2024
only_2024 = cols_2024 - cols_2025

print(f"Common columns ({len(common)}): {sorted(common)}")
print(f"\nOnly in 2025+ ({len(only_2025)}): {sorted(only_2025)}")
print(f"\nOnly in Through 2024 ({len(only_2024)}): {sorted(only_2024)}")

Common columns (13): ['COMMENTS', 'CRAB TRAP/DITM', 'DATE OF REQUIREMENT', 'DATE UPDATED', 'DEAL NAME', 'INTERNAL NOTES', 'MARKET', 'REQUIRED SF (HIGH)', 'REQUIRED SF (LOW)', 'TENANT REP BROKER CONTACT', 'TENANT REP COMPANY', 'TIMING', 'USE']

Only in 2025+ (8): ['ABSORBTION STATUS', 'ASSIGNED BROKER', 'BUILDINGS SENT', 'INDUSTRY', 'SOURCE', 'STATUS', 'Timing', 'Timing Trim']

Only in Through 2024 (3): ['', ' ', 'LEAD BROKER']


In [22]:
# Standardize Tab 0 (2025+) data
def standardize_tab0(df):
    """Standardize Tab 0 (2025+) data to common format"""
    df_std = pd.DataFrame()
    df_std['date'] = pd.to_datetime(df['DATE OF REQUIREMENT'], errors='coerce')
    df_std['company'] = df.get('COMPANY', '')
    df_std['sf_low'] = pd.to_numeric(df['REQUIRED SF (LOW)'], errors='coerce')
    df_std['sf_high'] = pd.to_numeric(df['REQUIRED SF (HIGH)'], errors='coerce')
    df_std['industry'] = df.get('INDUSTRY', '')
    df_std['market'] = df.get('MARKET', '')
    df_std['use_market'] = df.get('USE MARKET', '')
    df_std['status'] = df.get('STATUS', 'Active')
    df_std['source_tab'] = '2025+'
    return df_std

# Standardize Tab 1 (Through 2024) data
def standardize_tab1(df):
    """Standardize Tab 1 (Through 2024) data to common format"""
    df_std = pd.DataFrame()
    
    # Try to find date column - could be named differently
    date_cols = [col for col in df.columns if 'DATE' in col.upper()]
    if date_cols:
        df_std['date'] = pd.to_datetime(df[date_cols[0]], errors='coerce')
    else:
        df_std['date'] = pd.NaT
    
    # Map other columns (adjust these based on what you see in Tab 1)
    df_std['company'] = df.get('COMPANY', df.get('Company', ''))
    
    # SF columns - handle various possible names and formats
    sf_low_col = next((col for col in df.columns if 'SF' in col.upper() and 'LOW' in col.upper()), None)
    sf_high_col = next((col for col in df.columns if 'SF' in col.upper() and 'HIGH' in col.upper()), None)
    
    if sf_low_col:
        df_std['sf_low'] = pd.to_numeric(
            df[sf_low_col].astype(str).str.replace(',', '').str.replace('$', ''),
            errors='coerce'
        )
    else:
        df_std['sf_low'] = np.nan
    
    if sf_high_col:
        df_std['sf_high'] = pd.to_numeric(
            df[sf_high_col].astype(str).str.replace(',', '').str.replace('$', ''),
            errors='coerce'
        )
    else:
        df_std['sf_high'] = np.nan
    
    df_std['industry'] = df.get('INDUSTRY', df.get('Industry', ''))
    df_std['market'] = df.get('MARKET', df.get('Market', ''))
    df_std['use_market'] = df.get('USE MARKET', df.get('Use Market', ''))
    df_std['status'] = df.get('STATUS', df.get('Status', 'Active'))
    df_std['source_tab'] = 'Through 2024'
    
    return df_std

# Standardize both datasets
df_std_2025 = standardize_tab0(df_2025_plus)
df_std_2024 = standardize_tab1(df_through_2024)

print(f"Tab 0 standardized: {len(df_std_2025)} rows")
print(f"  Date range: {df_std_2025['date'].min()} to {df_std_2025['date'].max()}")
print(f"  Valid SF records: {df_std_2025['sf_low'].notna().sum()}")

print(f"\nTab 1 standardized: {len(df_std_2024)} rows")
print(f"  Date range: {df_std_2024['date'].min()} to {df_std_2024['date'].max()}")
print(f"  Valid SF records: {df_std_2024['sf_low'].notna().sum()}")

Tab 0 standardized: 739 rows
  Date range: 2024-08-08 00:00:00 to 2026-01-22 00:00:00
  Valid SF records: 737

Tab 1 standardized: 6971 rows
  Date range: 2012-06-09 00:00:00 to 2024-11-26 00:00:00
  Valid SF records: 6610


C:\Users\NLin\AppData\Local\Temp\ipykernel_23364\586130919.py:5: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

C:\Users\NLin\AppData\Local\Temp\ipykernel_23364\586130919.py:24: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [23]:
# Combine datasets
df_combined = pd.concat([df_std_2024, df_std_2025], ignore_index=True)

# Calculate average SF
df_combined['sf_avg'] = (df_combined['sf_low'] + df_combined['sf_high']) / 2

# Filter out records with no SF data
df_combined = df_combined[df_combined['sf_avg'].notna()].copy()

# Filter to 2018 onwards
df_combined = df_combined[df_combined['date'] >= '2018-01-01'].copy()

print(f"Combined dataset: {len(df_combined)} rows")
print(f"Date range: {df_combined['date'].min()} to {df_combined['date'].max()}")
print(f"Total SF (avg): {df_combined['sf_avg'].sum():,.0f}")
print(f"\nSource breakdown:")
print(df_combined['source_tab'].value_counts())

df_combined.head()

Combined dataset: 5971 rows
Date range: 2018-01-02 00:00:00 to 2026-01-22 00:00:00
Total SF (avg): 86,659,946

Source breakdown:
source_tab
Through 2024    5234
2025+            737
Name: count, dtype: int64


,date,company,sf_low,sf_high,industry,market,use_market,status,source_tab,sf_avg
0,2024-11-26,,5000.0,10000.0,,C,,Active,Through 2024,7500.0
1,2024-11-25,,20000.0,25000.0,,"E, S",,Active,Through 2024,22500.0
2,2024-11-25,,2000.0,4000.0,,NW,,Active,Through 2024,3000.0
3,2024-11-25,,6000.0,6000.0,,NW,,Active,Through 2024,6000.0
4,2024-11-22,,2000.0,4000.0,,"CBD, C",,Active,Through 2024,3000.0


In [24]:
# Aggregate data by month
monthly_data = df_combined.groupby(pd.Grouper(key='date', freq='ME')).agg({
    'sf_low': ['sum', 'count'],
    'sf_high': 'sum',
    'sf_avg': ['mean', 'median']
}).reset_index()

monthly_data.columns = [
    'date', 'sf_low_sum', 'count', 'sf_high_sum', 'sf_avg_mean', 'sf_avg_median'
]

print(f"Monthly data points: {len(monthly_data)}")
monthly_data.head()

Monthly data points: 97


,date,sf_low_sum,count,sf_high_sum,sf_avg_mean,sf_avg_median
0,2018-01-31,941000.0,95,1415400.0,12402.105263,8500.0
1,2018-02-28,1070600.0,100,1458400.0,12645.000000,5125.0
2,2018-03-31,451200.0,68,625900.0,7919.852941,5000.0
3,2018-04-30,684450.0,75,1032800.0,11448.333333,5500.0
4,2018-05-31,1341300.0,96,1661600.0,15640.104167,5500.0


In [25]:
# Chart styling
COLORS = {
    'background': '#FFFFFF',
    'text': '#2C3E50',
    'blue': '#00008B',
    'orange': '#DAA520',
    'light_gray': '#F8F9F9'
}

os.makedirs('charts/office', exist_ok=True)

## Chart 1: Total SF Requirements (Low/High)

In [30]:
fig1 = px.line(
    monthly_data,
    x='date',
    y=['sf_low_sum', 'sf_high_sum'],
    title='Monthly Total Square Footage Requirements (Combined Historical Data)',
    labels={
        'value': 'Square Footage',
        'date': 'Date',
        'sf_low_sum': 'Low Requirement (sqft)',
        'sf_high_sum': 'High Requirement (sqft)'
    },
    color_discrete_sequence=[COLORS['orange'], COLORS['blue']]
)
# Manually update the legend names if necessary
for i, trace_name in enumerate(['Low Requirement (sqft)', 'High Requirement (sqft)']):
    fig1.data[i].name = trace_name

fig1.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
    title={'font': dict(family=AQUILA_FONT, size=24, color=AQUILA_COLORS[0])},
    xaxis=dict(
        gridcolor=COLORS['light_gray'],
        showgrid=True,
        showline=True,
        linecolor='lightgrey'
    ),
    yaxis=dict(
        gridcolor=COLORS['light_gray'],
        showline=True,
        linecolor='lightgrey'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=100, b=50, l=50, r=50),
    height=550  # Increase the chart height here (default ~450)
)

fig1.write_html('charts/office/requirements_sf_total.html')
print('✓ Saved charts/office/requirements_sf_total.html')
fig1.show()

✓ Saved charts/requirements_sf_total.html


## Chart 2: Average SF Metrics (Dual-Axis)

In [31]:
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=monthly_data['date'],
    y=monthly_data['sf_avg_mean'],
    mode='lines+markers',
    name='Avg SF (Mean)',
    line=dict(color=COLORS['orange'])
))

fig2.add_trace(go.Scatter(
    x=monthly_data['date'],
    y=monthly_data['sf_avg_median'],
    mode='lines+markers',
    name='Avg SF (Median)',
    line=dict(color=COLORS['blue'])
))

fig2.add_trace(go.Bar(
    x=monthly_data['date'],
    y=monthly_data['count'],
    name='Record Count',
    yaxis='y2',
    marker_color=COLORS['text'],
    opacity=0.3
))

fig2.update_layout(
    title={
        'text': 'Monthly Average Square Footage Metrics (Combined Historical Data)',
        'font': dict(family=AQUILA_FONT, size=24, color=AQUILA_COLORS[0])
    },
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
    xaxis=dict(
        title='Date',
        gridcolor=COLORS['light_gray'],
        showgrid=True,
        showline=True,
        linecolor='lightgrey'
    ),
    yaxis=dict(
        title="Square Footage",
        gridcolor=COLORS['light_gray'],
        showline=True,
        linecolor='lightgrey'
    ),
    yaxis2=dict(
        title="Count",
        overlaying='y',
        side='right',
        titlefont=dict(family=AQUILA_FONT, size=12, color=COLORS['text']),
        tickfont=dict(family=AQUILA_FONT, size=12, color=COLORS['text'])
    ),
    legend=dict(orientation="h", y=-0.2),
    margin=dict(t=100, b=50, l=50, r=50)
)

fig2.write_html('charts/office/requirements_sf_avg.html')
print('✓ Saved charts/office/requirements_sf_avg.html')
fig2.show()

✓ Saved charts/requirements_sf_avg.html


## Chart 3: Demand by Industry (Donut Chart)

In [35]:
# Aggregate by industry
industry_data = (
    df_combined.dropna(subset=['sf_avg'])
    .groupby('industry')['sf_avg']
    .sum()
    .reset_index()
)
industry_data = industry_data[industry_data['sf_avg'] > 0]
industry_data = industry_data[industry_data['industry'].astype(str).str.strip() != '']

# Note how far back the data goes
if 'date' in df_combined.columns:
    min_date = df_combined['date'].min()
    max_date = df_combined['date'].max()
    date_range_note = f"(Data from {min_date:%b %Y} to {max_date:%b %Y})"
else:
    date_range_note = ""

# Top 7 + Other
industry_data_sorted = industry_data.sort_values(by='sf_avg', ascending=False)
top_n = 7
largest = industry_data_sorted.iloc[:top_n]
other = industry_data_sorted.iloc[top_n:]

if not other.empty:
    other_row = pd.DataFrame([{
        'industry': 'Other',
        'sf_avg': other['sf_avg'].sum()
    }])
    pie_data = pd.concat([largest, other_row], ignore_index=True)
else:
    pie_data = largest

# Extend color palette
industry_colors = (AQUILA_COLORS * ((len(pie_data) // len(AQUILA_COLORS)) + 1))[:len(pie_data)]

fig3 = go.Figure(
    data=[
        go.Pie(
            labels=pie_data['industry'],
            values=pie_data['sf_avg'],
            textinfo='label+percent',
            insidetextorientation='radial',
            hole=0.55,
            marker=dict(
                line=dict(color=COLORS['background'], width=2),
                colors=industry_colors
            )
        )
    ]
)

fig3.update_layout(
    title={
        'text': f'Tenant Demand by Industry (Combined Historical Data) {date_range_note}',
        'font': dict(family=AQUILA_FONT, size=24, color=AQUILA_COLORS[0])
    },
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
    showlegend=False,
    width=1000,
    height=650,
    margin=dict(t=100, b=80, l=50, r=50)
)

fig3.write_html('charts/office/requirements_sf_avg_by_industry.html')
print('✓ Saved charts/office/requirements_sf_avg_by_industry.html')
fig3.show()

✓ Saved charts/requirements_sf_avg_by_industry.html


## Chart 4: Requirements by Size Range

In [36]:
# Define size bins
bins = [0, 14000, 40000, 100000, float('inf')]
labels = ['0-14k', '15k-39k', '40k-99k', '100k+']

df_combined['size_range'] = pd.cut(df_combined['sf_avg'], bins=bins, labels=labels, right=False)

size_group = df_combined.groupby('size_range', observed=False).agg(
    total_sf=('sf_avg', 'sum'),
    count=('sf_avg', 'count')
).reset_index()

size_group['size_range'] = pd.Categorical(size_group['size_range'], categories=labels, ordered=True)
size_group = size_group.sort_values('size_range').reset_index(drop=True)

fig4 = go.Figure(
    data=[
        go.Bar(
            y=size_group['size_range'],
            x=size_group['total_sf'],
            orientation='h',
            marker_color=AQUILA_COLORS[:len(size_group)],
            text=size_group['count'],
            texttemplate='%{text} reqs',
            textposition='inside',
            textfont=dict(family=AQUILA_FONT, size=14),
            hovertemplate=(
                'Size Range: %{y}<br>'
                'Total SF: %{x:,.0f}<br>'
                'Count: %{text}<extra></extra>'
            ),
        )
    ]
)

fig4.update_layout(
    title={
        'text': 'Total Cumulative SF Requested by Size Range (Combined Historical Data)',
        'font': dict(family=AQUILA_FONT, size=22, color=AQUILA_COLORS[0])
    },
    xaxis=dict(
        title='Total Cumulative Requested SF',
        showgrid=True,
        showline=True,
        linecolor='lightgrey',
        linewidth=2
    ),
    yaxis=dict(
        title='Requirement Size Range (SF)',
        showline=True,
        linecolor='lightgrey',
        linewidth=2
    ),
    font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    width=820,
    height=500,
    margin=dict(t=80, b=80, l=120, r=50)
)

fig4.write_html('charts/office/requirements_by_size_range.html')
print('✓ Saved charts/office/requirements_by_size_range.html')
fig4.show()

✓ Saved charts/requirements_by_size_range.html


## NEW: Chart 5 - Requirements vs Absorption Comparison

In [42]:
# Fetch absorption data from Supabase
print("Fetching absorption data from Supabase...")

try:
    supabase = initialize_supabase_connection()
    print("  ✓ Connected to Supabase")
    
    # Query office market data for absorption
    all_records = []
    page = 0
    page_size = 1000
    
    while True:
        response = supabase.table('market_tables_office') \
            .select('quarter, total_net_absorption') \
            .gte('quarter', '2018 Q1') \
            .range(page * page_size, (page + 1) * page_size - 1) \
            .execute()
        
        batch = response.data
        all_records.extend(batch)
        
        if len(batch) < page_size:
            break
        page += 1

    df_absorption = pd.DataFrame(all_records)
    print(f"  ✓ Loaded {len(df_absorption)} absorption records")
    
    # Parse 'quarter' column like "2018 Q1" robustly
    def parse_quarter(qstr):
        import re
        m = re.match(r'(\d{4})\s*[Qq](\d)', str(qstr))
        if m:
            year = int(m.group(1))
            q = int(m.group(2))
            # pandas period to timestamp defaults to START of quarter
            return pd.Timestamp(f"{year}-{(q-1)*3+1:02d}-01")
        else:
            raise ValueError(f"Unknown quarter format: {qstr}")
    
    df_absorption['quarter'] = df_absorption['quarter'].apply(parse_quarter)
    df_absorption['total_net_absorption'] = pd.to_numeric(df_absorption['total_net_absorption'], errors='coerce')

    absorption_quarterly = df_absorption.groupby('quarter')['total_net_absorption'].sum().reset_index()
    absorption_quarterly.columns = ['quarter', 'absorption_sf']
    
    print(f"  ✓ Quarterly absorption data: {len(absorption_quarterly)} points")
    
except Exception as e:
    print(f"  ✗ Error: {e}")
    absorption_quarterly = None

Fetching absorption data from Supabase...
  ✓ Connected to Supabase
  ✓ Loaded 544 absorption records
  ✓ Quarterly absorption data: 32 points


In [43]:
# Create comparison chart if we have absorption data
if absorption_quarterly is not None:
    # Aggregate requirements by quarter
    df_combined['quarter'] = df_combined['date'].dt.to_period('Q').dt.to_timestamp()
    requirements_quarterly = df_combined.groupby('quarter').agg({
        'sf_avg': 'sum',
        'sf_low': 'sum',
        'sf_high': 'sum'
    }).reset_index()
    requirements_quarterly.columns = ['quarter', 'requirements_sf_avg', 'requirements_sf_low', 'requirements_sf_high']
    
    # Merge datasets
    comparison_df = pd.merge(
        requirements_quarterly,
        absorption_quarterly,
        on='quarter',
        how='outer'
    ).sort_values('quarter')
    
    print(f"Combined comparison data: {len(comparison_df)} quarters")
    
    # Create chart
    fig5 = go.Figure()
    
    # Requirements (average SF)
    fig5.add_trace(go.Scatter(
        x=comparison_df['quarter'],
        y=comparison_df['requirements_sf_avg'],
        mode='lines+markers',
        name='Requirements (Avg SF)',
        line=dict(color=AQUILA_COLORS[1], width=2.5),  # Gold
        marker=dict(size=8)
    ))
    
    # Absorption
    fig5.add_trace(go.Scatter(
        x=comparison_df['quarter'],
        y=comparison_df['absorption_sf'],
        mode='lines+markers',
        name='Absorption (Total SF)',
        line=dict(color=AQUILA_COLORS[0], width=2.5),  # Navy
        marker=dict(size=8)
    ))
    
    fig5.update_layout(
        title={
            'text': 'Office Requirements vs Absorption (Quarterly, 2018+)',
            'font': dict(family=AQUILA_FONT, size=24, color=AQUILA_COLORS[0])
        },
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
        xaxis=dict(
            title='Quarter',
            gridcolor='#e9e9ea',
            showgrid=True,
            showline=True,
            linecolor='lightgrey',
            linewidth=2
        ),
        yaxis=dict(
            title='Square Feet',
            gridcolor='#e9e9ea',
            showgrid=True,
            showline=True,
            linecolor='lightgrey',
            linewidth=2,
            tickformat=','
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.25,
            xanchor="center",
            x=0.5,
            font=dict(size=14)
        ),
        hovermode='x unified',
        height=600,
        margin=dict(t=100, b=100, l=80, r=50)
    )
    
    fig5.write_html('charts/office/requirements_vs_absorption_office.html')
    print('✓ Saved charts/office/requirements_vs_absorption_office.html')
    fig5.show()
else:
    print("⚠ Skipping comparison chart - no absorption data available")

Combined comparison data: 33 quarters
✓ Saved charts/requirements_vs_absorption_office.html


## NEW: Chart 6 - Rolling 12-Month Requirements with YoY Comparison

In [45]:
# Calculate rolling 12-month metrics
# Set index to date for rolling calculations
monthly_data_indexed = monthly_data.set_index('date').sort_index()

# Calculate rolling 12-month sum for average SF
# For average SF, we want the rolling mean (not sum)
rolling_data = pd.DataFrame()
rolling_data['current_avg_sf'] = monthly_data_indexed['sf_avg_mean'].rolling(window=12, min_periods=1).mean()
rolling_data['current_count'] = monthly_data_indexed['count'].rolling(window=12, min_periods=1).sum()

# Shift data by 12 months to get prior year
rolling_data['prior_avg_sf'] = rolling_data['current_avg_sf'].shift(12)
rolling_data['prior_count'] = rolling_data['current_count'].shift(12)

# Reset index for plotting
rolling_data = rolling_data.reset_index()

# Only show data where we have at least 12 months of history
rolling_data = rolling_data[rolling_data['date'] >= '2019-01-31'].copy()

print(f"Rolling 12-month data: {len(rolling_data)} points")
print(f"Date range: {rolling_data['date'].min()} to {rolling_data['date'].max()}")

# Create dual-axis chart
fig6 = go.Figure()

# Current year - Average SF (line, left axis)
fig6.add_trace(go.Scatter(
    x=rolling_data['date'],
    y=rolling_data['current_avg_sf'],
    mode='lines',
    name='Current Year Avg SF (12M Rolling)',
    line=dict(color=AQUILA_COLORS[0], width=2.5),  # Navy
    yaxis='y'
))

# Prior year - Average SF (line, left axis)
fig6.add_trace(go.Scatter(
    x=rolling_data['date'],
    y=rolling_data['prior_avg_sf'],
    mode='lines',
    name='Prior Year Avg SF (12M Rolling)',
    line=dict(color=AQUILA_COLORS[1], width=2.5, dash='dash'),  # Gold, dashed
    yaxis='y'
))

# Current year - Count (bar, right axis)
fig6.add_trace(go.Bar(
    x=rolling_data['date'],
    y=rolling_data['current_count'],
    name='Current Year Count (12M Rolling)',
    marker_color=AQUILA_COLORS[0],  # Navy
    opacity=0.3,
    yaxis='y2'
))

# Prior year - Count (bar, right axis)
fig6.add_trace(go.Bar(
    x=rolling_data['date'],
    y=rolling_data['prior_count'],
    name='Prior Year Count (12M Rolling)',
    marker_color=AQUILA_COLORS[1],  # Gold
    opacity=0.3,
    yaxis='y2'
))

fig6.update_layout(
    title={
        'text': 'Rolling 12-Month Requirements: Year-over-Year Comparison',
        'font': dict(family=AQUILA_FONT, size=24, color=AQUILA_COLORS[0])
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family=AQUILA_FONT, size=12, color=AQUILA_COLORS[0]),
    xaxis=dict(
        title='Date',
        gridcolor='#e9e9ea',
        showgrid=True,
        showline=True,
        linecolor='lightgrey',
        linewidth=2
    ),
    yaxis=dict(
        title='Average Square Feet (12M Rolling Mean)',
        gridcolor='#e9e9ea',
        showgrid=True,
        showline=True,
        linecolor='lightgrey',
        linewidth=2,
        tickformat=','
    ),
    yaxis2=dict(
        title='Count of Requirements (12M Rolling Sum)',
        overlaying='y',
        side='right',
        showgrid=False,
        showline=True,
        linecolor='lightgrey',
        linewidth=2
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    ),
    hovermode='x unified',
    height=650,
    margin=dict(t=100, b=120, l=80, r=80),
    barmode='overlay'
)

fig6.write_html('charts/office/requirements_yoy_rolling_12m.html')
print('✓ Saved charts/office/requirements_yoy_rolling_12m.html')
fig6.show()

Rolling 12-month data: 85 points
Date range: 2019-01-31 00:00:00 to 2026-01-31 00:00:00
✓ Saved charts/office/requirements_yoy_rolling_12m.html


print("="*80)
print("SUMMARY")
print("="*80)
print(f"\nCombined dataset:")
print(f"  - Total records: {len(df_combined)}")
print(f"  - Date range: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")
print(f"  - Tab 0 (2025+) records: {len(df_std_2025)}")
print(f"  - Tab 1 (Through 2024) records: {len(df_std_2024)}")
print(f"  - Total requirements SF (avg): {df_combined['sf_avg'].sum():,.0f}")
print(f"\nCharts generated:")
print(f"  ✓ charts/office/requirements_sf_total.html")
print(f"  ✓ charts/office/requirements_sf_avg.html")
print(f"  ✓ charts/office/requirements_sf_avg_by_industry.html")
print(f"  ✓ charts/office/requirements_by_size_range.html")
if absorption_quarterly is not None:
    print(f"  ✓ charts/office/requirements_vs_absorption_office.html")
print(f"  ✓ charts/office/requirements_yoy_rolling_12m.html")
print("\n" + "="*80)

In [46]:
print("="*80)
print("SUMMARY")
print("="*80)
print(f"\nCombined dataset:")
print(f"  - Total records: {len(df_combined)}")
print(f"  - Date range: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")
print(f"  - Tab 0 (2025+) records: {len(df_std_2025)}")
print(f"  - Tab 1 (Through 2024) records: {len(df_std_2024)}")
print(f"  - Total requirements SF (avg): {df_combined['sf_avg'].sum():,.0f}")

print(f"\nCharts generated:")
print(f"  ✓ charts/office/requirements_sf_total.html")
print(f"  ✓ charts/office/requirements_sf_avg.html")
print(f"  ✓ charts/office/requirements_sf_avg_by_industry.html")
print(f"  ✓ charts/office/requirements_by_size_range.html")
if absorption_quarterly is not None:
    print(f"  ✓ charts/office/requirements_vs_absorption_office.html")
print("\n" + "="*80)

SUMMARY

Combined dataset:
  - Total records: 5971
  - Date range: 2018-01-02 to 2026-01-22
  - Tab 0 (2025+) records: 739
  - Tab 1 (Through 2024) records: 6971
  - Total requirements SF (avg): 86,659,946

Charts generated:
  ✓ charts/office/requirements_sf_total.html
  ✓ charts/office/requirements_sf_avg.html
  ✓ charts/office/requirements_sf_avg_by_industry.html
  ✓ charts/office/requirements_by_size_range.html
  ✓ charts/office/requirements_vs_absorption_office.html

